# ELSA Depression Prediction
### MSc AI & Health — Group Assignment

**Goal:** Use Wave 6 features to predict whether a participant has depression at Wave 7.

**Target variable:** `hepsyde` — *Psychiatric problem: depression* (0 = not mentioned, 1 = mentioned)

**Models:** Random Forest + XGBoost (compared side by side)

## Cell 1 — Install & Import Libraries

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## Cell 2 — Load the Data

In [ ]:
FEATURE_COLS = [
    # Sociodemographic
    'DiSex',     # Sex (1=male, 2=female)
    'DiMar',     # Marital status
    'DiMaedu',   # Education level

    # Physical health
    'Hehelf',    # Self-rated health (1=excellent, 5=poor)
    'Heill',     # Long-standing illness
    'Helim',     # Health limits daily activities
    'HePain',    # Pain
    'HeSmk',     # Smoking
    'HeActa',    # Vigorous physical activity
    'HeActb',    # Moderate physical activity
    'HeActc',    # Mild physical activity

    # Prior wave mental health (Wave 6)
    'hepsyde',   # Had depression
    'hepsyan',   # Anxiety
    'hepsyem',   # Emotional problems
    'hepsyps',   # Psychosis
    'hepsymo',   # Mood Swings
    'hepsyma',   # Manic Depression
    'PScedA',
    'PScedB',
    'PScedC',
    'PScedD',
    'PScedE',
    'PScedF',
    'PScedG',
    'PScedH'
]

w6_cols = ['idauniq'] + FEATURE_COLS
w6 = pd.read_stata('wave_6_elsa_data_v2.dta', columns=w6_cols, convert_categoricals=False)

w7_cols = ['idauniq'] + FEATURE_COLS
w7 = pd.read_stata('wave_7_elsa_data.dta', columns=w7_cols, convert_categoricals=False)

print(f'Wave 6: {w6.shape[0]:,} participants, {w6.shape[1]} columns')
print(f'Wave 7: {w7.shape[0]:,} participants, {w7.shape[1]} columns')

print('\nWave 6 columns:', list(w6.columns))
print('\nWave 7 columns:', list(w7.columns))

## Cell 3 — Build the Target Variable (Wave 7 Depression)

`hepsyde` = 1 means the participant mentioned depression as a psychiatric problem at Wave 7.  
We treat this as our binary prediction target.

In [ ]:
# -1 (not applicable) likely means no psychiatric problems, so treat as 0
w7_target = w7[['idauniq', 'hepsyde']].copy()
w7_target['hepsyde'] = w7_target['hepsyde'].replace({-1: 0})

# Only treat genuinely missing codes as NaN
w7_target['hepsyde'] = w7_target['hepsyde'].replace([-2, -8, -9], np.nan)

w7_target = w7_target.dropna(subset=['hepsyde'])
w7_target['depressed_w7'] = w7_target['hepsyde'].astype(int)

print(f'Wave 7 participants with valid depression label: {len(w7_target):,}')
print(f"\nClass balance:")
print(f"  Depressed (hepsyde=1): {w7_target['depressed_w7'].sum()} ({w7_target['depressed_w7'].mean()*100:.1f}%)")
print(f"  Not depressed (hepsyde=0): {(w7_target['depressed_w7']==0).sum()} ({(1-w7_target['depressed_w7'].mean())*100:.1f}%)")

## Cell 4 — Select Features from Wave 6

We use Wave 6 data to predict Wave 7 outcomes — this is a **longitudinal prediction** setup.

Feature categories:
- **Sociodemographic:** sex, marital status, education
- **Social contact:** WhoSo1–5 (who participant socialises with)
- **Physical health:** self-rated health, illness, pain, smoking, activity
- **Prior mental health:** hepsyde, hepsyan etc. from Wave 6 (strongest predictors)

In [ ]:
FEATURE_LABELS = {
    'hepsyde': 'Has depression',
    'Heill': 'Long standing illness',
    'HePain': 'Often troubled with pain',
    'HeSmk': 'Ever smoked cigarettes',
    'DiSex': 'Sex',
    'DiMar': 'Marital status',
    'DiMaedu': 'Education level',
    'WhoSo1': 'Socialises with: children',
    'WhoSo2': 'Socialises with: friends',
    'PScedA': 'Felt depressed past week',
    'PScedB': 'Felt effort past week',
    'PScedC': 'Restless sleep past week',
    'PScedD': 'Happy past week',
    'PScedE': 'Felt lonely past week',
    'PScedF': 'Enjoyed life past week',
    'PScedG': 'Felt sad past week',
    'PScedH': 'Couldnt get going much past week',
    'Hehelf': 'Self-rated health',
    'HeActa': 'Vigorous activity',
    'HeActb': 'Frequency does moderate sports',
    'HeActc': 'Frequency does mild sports',
    'hepsyan': 'Has anxiety',
    'Helim': 'Long standing illness is limiting',
    'hepsyps': 'Has psychosis',
    'WhoSo1':'No-one other than interviewer in room',
    'WhoSo2':'Spouse/Partner in room',
    'WhoSo3':'Adult household member in room',
    'WhoSo4':'Child household member in room',
    'WhoSo5':'Non-household member in room',
    'hepsymo':'Has mood swings',
    'hepsyma':'Has manic depression',
    'hepsyem':'Has emotional problems'
}

readable_names = [FEATURE_LABELS.get(col, col) for col in FEATURE_COLS]

w6_features = w6[['idauniq'] + FEATURE_COLS].copy()

# Replace all ELSA missing codes with NaN
for col in FEATURE_COLS:
    w6_features[col] = w6_features[col].replace(MISSING_CODES, np.nan)

print(f'Wave 6 features extracted: {len(FEATURE_COLS)} features')
print(f'\nMissing data summary (% missing):')
missing_pct = (w6_features[FEATURE_COLS].isnull().sum() / len(w6_features) * 100).round(1)
missing_pct.index = readable_names
print(missing_pct[missing_pct > 0].to_string())

## Cell 5 — Merge Waves & Final Dataset

In [ ]:
# Inner join: keep only participants present in both waves with a valid target
df = pd.merge(
    w6_features,
    w7_target[['idauniq', 'depressed_w7']],
    on='idauniq',
    how='inner'
)

print(f'Final dataset: {len(df):,} participants')
print(f'Depressed at Wave 7: {df["depressed_w7"].sum()} ({df["depressed_w7"].mean()*100:.1f}%)')
print(f'Not depressed: {(df["depressed_w7"]==0).sum()} ({(1-df["depressed_w7"].mean())*100:.1f}%)')

# Separate features and target
X = df[FEATURE_COLS]
y = df['depressed_w7']

print(f'\nFeature matrix shape: {X.shape}')

## Cell 6 — Train / Test Split

In [ ]:
# Stratified split preserves class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,
    stratify=y
)

print(f'Training set: {len(X_train):,} participants ({y_train.mean()*100:.1f}% depressed)')
print(f'Test set:     {len(X_test):,} participants ({y_test.mean()*100:.1f}% depressed)')

In [ ]:
MISSING_CODES = [-1, -2, -8, -9]

for col in ['PScedA','PScedB','PScedC','PScedD','PScedE','PScedF','PScedG','PScedH']:
    w7[col] = w7[col].replace(MISSING_CODES, np.nan)

negative_items = ['PScedA','PScedB','PScedC','PScedE','PScedG','PScedH']
positive_items = ['PScedD','PScedF']

for col in negative_items:
    # Yes=1 means depressed, recode to 1; No=2 means not, recode to 0
    w7[col + '_score'] = w7[col].map({1: 1, 2: 0})

for col in positive_items:
    # Yes=1 means NOT depressed, recode to 0; No=2 means depressed, recode to 1
    w7[col + '_score'] = w7[col].map({1: 0, 2: 1})

score_cols = [c + '_score' for c in negative_items + positive_items]
w7['cesd_total'] = w7[score_cols].sum(axis=1, min_count=8)  # min_count=8 ensures all items answered

# Shades of Blue and Gray: A Comparison of the Center for Epidemiologic Studies Depression Scale and the Composite International Diagnostic Interview for Assessment of Depression Syndrome in Later Life.
# Available from: https://www.researchgate.net/publication/333293635_Shades_of_Blue_and_Gray_A_Comparison_of_the_Center_for_Epidemiologic_Studies_Depression_Scale_and_the_Composite_International_Diagnostic_Interview_for_Assessment_of_Depression_Syndrome_in_Later_Life#fullTextFileContent
# Prior work in the HRS has identified that a threshold of ≥4 corresponds to the commonly used cutoff score of 16 in the original 20-item CESD (Steffick, 2000).
# Therefore, in this study, we explored three thresholds (≥3, ≥4, and ≥5) to indicate depression case status using the CESD-8.

# Either a threshold of 3 or 4 is justifiable based on the above source. Using a threshold of 3 here but we could try both.
w7['depressed_w7'] = (w7['cesd_total'] >= 3).astype(float)
print(w7['depressed_w7'] )

## Cell 7 — Train Random Forest

Key settings:
- `n_estimators=300` — 300 trees in the forest
- `class_weight='balanced'` — automatically handles class imbalance
- `SimpleImputer` — fills missing values with the median before training

In [ ]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]
rf_auc  = roc_auc_score(y_test, rf_prob)

print('Random Forest trained.')
print(f'ROC-AUC on test set: {rf_auc:.3f}')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=['Not Depressed', 'Depressed']))

## Cell 8 — Train XGBoost

XGBoost is a **gradient boosting** model — instead of building trees independently (like Random Forest), it builds each tree to correct the errors of the previous one. Often more accurate, but slightly harder to tune.

In [ ]:
# Calculate scale_pos_weight to handle class imbalance (equivalent to class_weight='balanced')
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f'Class imbalance ratio (scale_pos_weight): {scale:.2f}')

xgb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale,   # handles class imbalance
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)
xgb_prob = xgb_pipeline.predict_proba(X_test)[:, 1]
xgb_auc  = roc_auc_score(y_test, xgb_prob)

print('\nXGBoost trained.')
print(f'ROC-AUC on test set: {xgb_auc:.3f}')
print('\nClassification Report:')
print(classification_report(y_test, xgb_pred, target_names=['Not Depressed', 'Depressed']))

## Cell 9 — Cross Validation (More Robust Evaluation)

A single train/test split can be lucky or unlucky. 5-fold cross-validation gives a more reliable estimate of true model performance.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv  = cross_val_score(rf_pipeline,  X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
xgb_cv = cross_val_score(xgb_pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print('5-Fold Cross-Validation AUC:')
print(f'  Random Forest: {rf_cv.mean():.3f}  (+/- {rf_cv.std():.3f})')
print(f'  XGBoost:       {xgb_cv.mean():.3f}  (+/- {xgb_cv.std():.3f})')

## Cell 10 — Plot Results: Confusion Matrices & ROC Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ELSA Depression Prediction — Model Comparison', fontsize=14, fontweight='bold')

# --- Confusion Matrix: Random Forest ---
ConfusionMatrixDisplay(
    confusion_matrix(y_test, rf_pred),
    display_labels=['Not Depressed', 'Depressed']
).plot(ax=axes[0, 0], colorbar=False, cmap='Blues')
axes[0, 0].set_title('Random Forest — Confusion Matrix')

# --- Confusion Matrix: XGBoost ---
ConfusionMatrixDisplay(
    confusion_matrix(y_test, xgb_pred),
    display_labels=['Not Depressed', 'Depressed']
).plot(ax=axes[0, 1], colorbar=False, cmap='Oranges')
axes[0, 1].set_title('XGBoost — Confusion Matrix')

# --- ROC Curve: Both models ---
for prob, label, color, auc in [
    (rf_prob,  'Random Forest', 'steelblue', rf_auc),
    (xgb_prob, 'XGBoost',       'darkorange', xgb_auc),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[1, 0].plot(fpr, tpr, lw=2, label=f'{label} (AUC={auc:.3f})', color=color)

axes[1, 0].plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
axes[1, 0].set_xlabel('False Positive Rate')
axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].set_title('ROC Curve Comparison')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# --- Feature Importance: Random Forest ---
rf_model = rf_pipeline.named_steps['model']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:12]

axes[1, 1].barh(
    range(len(indices)),
    importances[indices][::-1],
    color='steelblue', alpha=0.8
)
axes[1, 1].set_yticks(range(len(indices)))
axes[1, 1].set_yticklabels([readable_names[i] for i in indices][::-1], fontsize=9)
axes[1, 1].set_xlabel('Feature Importance (Gini)')
axes[1, 1].set_title('Top 12 Features — Random Forest')
axes[1, 1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('elsa_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as elsa_results.png')

## Cell 11 — Feature Importance: XGBoost

In [ ]:
xgb_model = xgb_pipeline.named_steps['model']
xgb_importances = xgb_model.feature_importances_
xgb_indices = np.argsort(xgb_importances)[::-1][:12]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    range(len(xgb_indices)),
    xgb_importances[xgb_indices][::-1],
    color='darkorange', alpha=0.8
)
ax.set_yticks(range(len(xgb_indices)))
ax.set_yticklabels([readable_names[i] for i in xgb_indices][::-1], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 12 Features — XGBoost')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('elsa_xgb_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 12 — Summary Table

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'Accuracy':  [accuracy_score(y_test, rf_pred),  accuracy_score(y_test, xgb_pred)],
    'Precision': [precision_score(y_test, rf_pred), precision_score(y_test, xgb_pred)],
    'Recall':    [recall_score(y_test, rf_pred),    recall_score(y_test, xgb_pred)],
    'F1 Score':  [f1_score(y_test, rf_pred),        f1_score(y_test, xgb_pred)],
    'ROC-AUC':   [rf_auc,                           xgb_auc],
    'CV AUC (mean)': [rf_cv.mean(),                 xgb_cv.mean()],
}).round(3)

results = results.set_index('Model')
print(results.to_string())
results

# Cell 13 - SHAP Analysis

The Gini importance only shows the magnitudes.

SHAP has the advantage of showing the direction of each feature's effect, not just its magnitude.

For a guide to SHAP, see these two chapters in the Interpretable ML book:

https://christophm.github.io/interpretable-ml-book/shapley.html

https://christophm.github.io/interpretable-ml-book/shap.html

In [ ]:
import shap

# Apply SHAP to Random Forest model
# Ensure the imputer is fitted and transform X_test
X_test_rf_numpy = rf_pipeline.named_steps['imputer'].transform(X_test)

# Convert to DataFrame with feature names for SHAP consistency
X_test_rf = pd.DataFrame(X_test_rf_numpy, columns=FEATURE_COLS)
print(X_test_rf)

shap_explainer = shap.TreeExplainer(rf_pipeline.named_steps['model'])

# Calculate SHAP values using the DataFrame
rf_shap_values = shap_explainer.shap_values(X_test_rf)
print(rf_shap_values)

In [ ]:
shap.summary_plot(rf_shap_values[:, :, 1], X_test_rf, show=False, feature_names=readable_names)
plt.title('SHAP Summary Plot (Random Forest)')
plt.tight_layout()
plt.savefig('rf_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply SHAP to XGBoost model
X_test_xgb = xgb_pipeline.named_steps['imputer'].transform(X_test)
print(X_test_xgb)

shap_explainer = shap.TreeExplainer(xgb_pipeline.named_steps['model'])

xgb_shap_values = shap_explainer.shap_values(X_test)
print(xgb_shap_values)


In [ ]:
X_test_xgb_df = pd.DataFrame(X_test_xgb, columns=FEATURE_COLS)

shap.summary_plot(xgb_shap_values, X_test_xgb_df, show=False, feature_names=readable_names)
plt.title('SHAP Summary Plot (XGBoost)')
plt.tight_layout()
plt.savefig('xgb_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()